# TP Mentoré : Pipeline de Nettoyage et Réconciliation Multi-Sources (2h)
## Modélisation Retail & Audit de Cohérence Comptable

### Contexte métier
Vous intégrez le pôle Data de l'enseigne. Les systèmes de caisse, le référentiel magasin et la table de remises marketing sont déconnectés. Vous devez réconcilier ces sources hétérogènes, traquer les incohérences de schémas et valider les métriques financières.

---
### Les 6 missions du TP :
1. **Mission 1 — Ingestion & Résolution des conflits de typage** (20 min)
2. **Mission 2 — Réconciliation multi-tables & Détection des orphelins** (25 min)
3. **Mission 3 — Application des règles de remise & Calculs financiers** (25 min)
4. **Mission 4 — Audit comptable & Tests d'intégrité stricts** (20 min)
5. **Mission 5 — Métriques de contribution locale (`transform`)** (15 min)
6. **Mission 6 — Vue décisionnelle pivotée & Export Parquet** (15 min)

In [ ]:
import pandas as pd
import numpy as np
import os

print("Environnement prêt pour le pipeline consolidé !")

## Mission 1 : Ingestion & Résolution des conflits de typage (20 min)

Dans `data/` se trouvent : `magasins.csv`, `produits.csv`, `transactions.csv` et `promotions.csv`.

### Consignes :
1. Chargez les tables. Attention : dans `transactions.csv`, la colonne `magasin_id` contient des espaces parasites ou du texte, alors que dans `magasins.csv`, il s'agit d'entiers.
2. Nettoyez et forcez `magasin_id` au format numérique (`int` ou `float`) dans les deux tables avant toute tentative de jointure.
3. Supprimez les doublons stricts sur `transactions.csv` et imputez les quantités manquantes par `1`.
4. Convertissez `date_heure` en format `datetime64`.

In [ ]:
# TODO : Chargement, harmonisation des clés et typage
df_magasins = ...
df_produits = ...
df_transactions = ...
df_promos = ...

# Nettoyages et conversions
# ...

print("Schémas harmonisés prêts pour la réconciliation.")

In [ ]:
assert pd.api.types.is_numeric_dtype(df_transactions["magasin_id"]), "magasin_id dans transactions doit être numérique."
assert pd.api.types.is_numeric_dtype(df_magasins["magasin_id"]), "magasin_id dans magasins doit être numérique."
assert df_transactions["quantite"].isna().sum() == 0, "Il reste des quantités nulles."
assert pd.api.types.is_datetime64_any_dtype(df_transactions["date_heure"]), "date_heure doit être au format datetime."
print("✅ Mission 1 validée !")

## Mission 2 : Réconciliation multi-tables & Détection des orphelins (25 min)

### Consignes :
1. Joignez `df_transactions` avec `df_magasins` (`how='left'`, `indicator=True`). Détectez et quantifiez les transactions rattachées à un magasin inexistant (`_merge == 'left_only'`) dans `nb_magasins_orphelins`.
2. Ne conservez que les transactions avec magasin valide, supprimez la colonne `_merge`.
3. Joignez le résultat avec `df_produits` (`on='produit_id'`, `how='left'`).
4. Joignez ensuite avec `df_promos` (`on='rayon'`, `how='left'`). Pour les rayons sans promotion, imputez le taux de remise `taux_remise` par `0.0`.
5. Stockez le résultat réconcilié dans `df_faits`.

In [ ]:
# TODO : Chaînage des 3 jointures et audit des orphelins
nb_magasins_orphelins = ...
df_faits = ...

print(f"Transactions orphelines éliminées : {nb_magasins_orphelins}")
print("Colonnes consolidées :", df_faits.columns.tolist())

In [ ]:
assert nb_magasins_orphelins > 0, "Des orphelins de magasin doivent être détectés."
assert "taux_remise" in df_faits.columns, "La colonne taux_remise issue des promotions est absente."
assert df_faits["taux_remise"].isna().sum() == 0, "Les remises manquantes doivent être imputées par 0.0."
assert "ville" in df_faits.columns and "prix_unitaire_ht" in df_faits.columns, "Les dimensions magasin et produit doivent être présentes."
print("✅ Mission 2 validée !")

## Mission 3 : Règles de gestion & Calculs financiers (25 min)

### Consignes :
1. Isolez les ventes effectives dans `df_ventes` : filtrez pour ne garder que `statut == 'VALIDE'` et `quantite > 0`.
2. Calculez les métriques comptables ligne par ligne :
   - `montant_brut_ht` = `quantite * prix_unitaire_ht`
   - `remise_appliquee` = `montant_brut_ht * taux_remise` (arrondi à 2 décimales)
   - `montant_net_ht` = `montant_brut_ht - remise_appliquee`
   - `montant_ttc` = `montant_net_ht * (1 + tva)` (arrondi à 2 décimales)
3. Extrayez le mois de la vente sous forme textuelle ou numérique dans une nouvelle colonne `mois` (ex. `df_ventes['date_heure'].dt.month`).

In [ ]:
# TODO : Filtrage métier et calculs de facturation
df_ventes = ...

# Ajout des métriques de facturation
# ...

df_ventes[["transaction_id", "montant_brut_ht", "remise_appliquee", "montant_ttc", "mois"]].head()

In [ ]:
assert (df_ventes["statut"] == "VALIDE").all(), "Des transactions non valides subsistent."
assert (df_ventes["quantite"] > 0).all(), "Des quantités <= 0 subsistent."
assert (df_ventes["montant_net_ht"] <= df_ventes["montant_brut_ht"]).all(), "Le montant net ne peut dépasser le montant brut."
assert "mois" in df_ventes.columns, "La colonne 'mois' est manquante."
print("✅ Mission 3 validée !")

## Mission 4 : Audit comptable & Tests d'intégrité (20 min)

Vérifiez la concordance exacte de vos chiffres comme un auditeur de données SQL :

### Consignes :
1. Stockez le chiffre d'affaires total TTC de l'ensemble du réseau dans `ca_total_reseau`.
2. Calculez le CA TTC réalisé par le magasin de **Paris Haussmann** au mois de **Février (mois == 2)** dans `ca_paris_fevrier`.
3. Calculez le montant cumulé total des remises accordées aux clients sur l'ensemble de la période dans `total_remises_accordees`.

In [ ]:
# TODO : Requêtes d'audit financier ciblées
ca_total_reseau = ...
ca_paris_fevrier = ...
total_remises_accordees = ...

print(f"CA Total Réseau TTC      : {ca_total_reseau:,.2f} €")
print(f"CA Paris (Février) TTC   : {ca_paris_fevrier:,.2f} €")
print(f"Total Remises Accordées  : {total_remises_accordees:,.2f} €")

In [ ]:
assert isinstance(ca_total_reseau, (float, np.floating)), "ca_total_reseau doit être un float."
assert isinstance(ca_paris_fevrier, (float, np.floating)), "ca_paris_fevrier doit être un float."
assert ca_paris_fevrier < ca_total_reseau, "Incohérence sur le périmètre de CA."
assert total_remises_accordees > 0, "Des remises auraient dû être comptabilisées."
print("✅ Mission 4 validée : Concordance comptable certifiée !")

## Mission 5 : Métriques de contribution locale via `transform` (15 min)

La direction souhaite analyser le poids relatif de chaque ticket au sein de son propre magasin.

### Consignes :
1. Utilisez `groupby('nom_magasin')['montant_ttc'].transform('sum')` pour calculer le CA total du magasin correspondant sur chaque ligne, et stockez-le dans `ca_magasin_global`.
2. Calculez dans une colonne `part_ticket_pct` le poids du ticket par rapport à son magasin : `(df_ventes['montant_ttc'] / df_ventes['ca_magasin_global']) * 100` (arrondi à 3 décimales).

In [ ]:
# TODO : Calcul par partition sans agrégation (transform)
df_ventes["ca_magasin_global"] = ...
df_ventes["part_ticket_pct"] = ...

df_ventes[["nom_magasin", "montant_ttc", "ca_magasin_global", "part_ticket_pct"]].head()

In [ ]:
assert "part_ticket_pct" in df_ventes.columns, "La colonne part_ticket_pct est absente."
assert (df_ventes["part_ticket_pct"] <= 100.0).all(), "Une part relative ne peut dépasser 100%."
assert (df_ventes["part_ticket_pct"] > 0.0).all(), "Les parts de tickets doivent être positives."
print("✅ Mission 5 validée !")

## Mission 6 : Vue décisionnelle pivotée & Export Parquet (15 min)

### Consignes :
1. Construisez `df_matrice_ca` via `pivot_table` représentant la somme des `montant_ttc` avec :
   - En lignes : `nom_magasin`
   - En colonnes : `mois`
   - Fonction : `'sum'`
   - Remplacement des vides par `0.0`.
2. Exportez `df_ventes` au format Parquet compressé : `data/faits_ventes_final.parquet`.

In [ ]:
# TODO : Table croisée finale et export Parquet
df_matrice_ca = ...

df_ventes.to_parquet("data/faits_ventes_final.parquet", compression="snappy", index=False)

print("Matrice décisionnelle Ventes (Magasins x Mois) :")
df_matrice_ca

In [ ]:
assert isinstance(df_matrice_ca, pd.DataFrame), "df_matrice_ca doit être un DataFrame."
assert os.path.exists("data/faits_ventes_final.parquet"), "Le fichier Parquet final est introuvable."
print("🎉 FÉLICITATIONS ! Vous avez brillamment relevé le défi du TP Pipeline !")